# Task 1: Content-Based Recommendation Engine

Recommend the top-$K$ most similar titles for any movie or TV show in the catalog using:
1. An **entity-token metadata soup** built from genres, directors, countries, and rating.
2. **TF-IDF** weighting (sublinear term frequency).
3. **Cosine similarity** ranking.

## 1. Method

**TF-IDF** for term $t$ in document $d$ of corpus $D$:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \left(\ln\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|} + 1\right)$$

**Cosine similarity** between vectors $\mathbf{u}, \mathbf{v}$:
$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$
Because TF-IDF rows are $L_2$-normalized, this reduces to a dot product.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data

df = get_preprocessed_data('../data/Dataset.csv')
print(f"Loaded {len(df):,} titles with {len(df.columns)} columns.")
df[['title', 'type', 'director', 'country', 'rating', 'listed_in']].head()

Loaded 8,790 titles with 19 columns.


,title,type,director,country,rating,listed_in
0,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,PG-13,Documentaries
1,Ganglands,TV Show,Julien Leclercq,France,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,TV Show,Mike Flanagan,United States,TV-MA,"TV Dramas, TV Horror, TV Mysteries"
3,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,TV-PG,"Children & Family Movies, Comedies"
4,Sankofa,Movie,Haile Gerima,United States,TV-MA,"Dramas, Independent Movies, International Movies"


## 2. Metadata Soup

Each genre, director, country, and rating becomes **one whole token** (e.g. `d_kirsten_johnson`),
so a director named *Johnson* never matches an unrelated title containing the word *Johnson*.
Genres are repeated so they carry the most weight.

In [2]:
from src.recommender import MovieRecommender

recommender = MovieRecommender().fit(df)
soups = recommender._prepare_metadata_soup(df.head(3))
for title, soup in zip(df['title'].head(3), soups):
    print(f"[{title}] {soup}")
print(f"\nTF-IDF matrix: {recommender.tfidf_matrix.shape}")

[Dick Johnson Is Dead] g_documentaries g_documentaries d_kirsten_johnson c_united_states r_pg_13
[Ganglands] g_crime_tv_shows g_international_tv_shows g_tv_action_adventure g_crime_tv_shows g_international_tv_shows g_tv_action_adventure d_julien_leclercq c_france r_tv_ma
[Midnight Mass] g_tv_dramas g_tv_horror g_tv_mysteries g_tv_dramas g_tv_horror g_tv_mysteries d_mike_flanagan c_united_states r_tv_ma

TF-IDF matrix: (8790, 5128)


## 3. Recommendations

In [3]:
recommender.get_recommendations('Dick Johnson Is Dead', top_n=5)[['title', 'type', 'listed_in', 'similarity_score']]

,title,type,listed_in,similarity_score
0,Murder to Mercy: The Cyntoia Brown Story,Movie,Documentaries,0.4791
1,The Irishman: In Conversation,Movie,Documentaries,0.4791
2,Liberated: The New Sexual Revolution,Movie,Documentaries,0.4791
3,Free to Play,Movie,Documentaries,0.4688
4,Creating The Queen's Gambit,Movie,Documentaries,0.4688


In [4]:
recommender.get_recommendations('Midnight Mass', top_n=5)[['title', 'type', 'listed_in', 'similarity_score']]

,title,type,listed_in,similarity_score
0,Brand New Cherry Flavor,TV Show,"TV Dramas, TV Horror, TV Mysteries",0.8698
1,The Haunting of Bly Manor,TV Show,"TV Dramas, TV Horror, TV Mysteries",0.8698
2,Ratched,TV Show,"TV Dramas, TV Horror, TV Mysteries",0.8698
3,The Haunting of Hill House,TV Show,"TV Dramas, TV Horror, TV Mysteries",0.8698
4,The Originals,TV Show,"TV Dramas, TV Horror, TV Mysteries",0.8508


In [5]:
recommender.get_recommendations('Midnight Mass', top_n=5, content_type_filter='Movie')[['title', 'type', 'listed_in', 'similarity_score']]

,title,type,listed_in,similarity_score
0,Gerald's Game,Movie,"Horror Movies, Thrillers",0.3563
1,Hush,Movie,"Horror Movies, Thrillers",0.3311
2,Before I Wake,Movie,"Horror Movies, Thrillers",0.3278
3,Bad Trip,Movie,Comedies,0.0879
4,Still LAUGH-IN: The Stars Celebrate,Movie,Comedies,0.0879


## 4. Genre Overlap Check

A simple quality proxy: the share of recommendations that share at least one genre with the query title.

In [6]:
rng = np.random.default_rng(42)
sample_titles = rng.choice(df['title'].unique(), size=200, replace=False)

def genre_set(s):
    return {g.strip() for g in s.split(',')}

overlaps = []
for t in sample_titles:
    query_genres = genre_set(df.loc[df['title'] == t, 'listed_in'].iloc[0])
    recs = recommender.get_recommendations(t, top_n=5)
    overlaps += [bool(query_genres & genre_set(g)) for g in recs['listed_in']]

print(f"Top-5 genre overlap over {len(sample_titles)} random titles: {np.mean(overlaps):.1%}")

Top-5 genre overlap over 200 random titles: 98.8%


## 5. Notes

- A single sparse dot product against the full matrix keeps queries fast at this catalog size.
- For much larger catalogs, approximate nearest-neighbour search (FAISS, HNSW) would replace the full scan.